# Step 12.3 — Stable Attention Computation：Softmax 为什么需要数值稳定？Scaling 到底帮助什么？

Step 12.2 已经得到 masked scores：

$$
S_{masked}
=
\frac{QK^T}{\sqrt{d_k}}+M,
$$

其中 future positions 已经被写成 `-∞`。

接下来需要计算：

$$
A=\operatorname{softmax}(S_{masked},\ axis=-1).
$$

数学公式看起来很简单，但如果直接计算 `exp(score)`，较大的 logits 可能造成 overflow，较小值也可能发生 underflow。

本节只讲 stable softmax 的核心数学直觉、mask 与稳定计算怎样兼容，以及 scaling 真正帮助的数值/优化问题；不进入 Step 12.4 的完整 causal attention head。


## 1. Naive Softmax 的问题：指数函数增长太快

对一行 logits：

$$
z=[z_0,z_1,\ldots,z_{T-1}],
$$

softmax 定义为：

$$
\operatorname{softmax}(z_i)
=
\frac{e^{z_i}}{\sum_j e^{z_j}}.
$$

如果 logits 很大，例如：

$$
z=[1000,999,998],
$$

数学上 softmax 完全有定义，因为真正重要的是这些 logits 的**相对差值**。

但计算机若直接尝试：

```text
exp(1000)
exp(999)
exp(998)
```

有限精度浮点数可能无法表示这些巨大数值，于是产生 `inf`（overflow）。

接着朴素公式可能出现：

```text
inf / inf
```

从而得到 `nan`。

所以问题不是 softmax 数学本身有错，而是**直接按照最朴素公式计算不适合有限精度浮点数**。


## 2. 关键数学：Softmax 对“整行减同一个常数”不变

对任意常数 `c`：

$$
\frac{e^{z_i-c}}{\sum_j e^{z_j-c}}
=
\frac{e^{z_i}e^{-c}}{\sum_j e^{z_j}e^{-c}}
=
\frac{e^{z_i}}{\sum_j e^{z_j}}.
$$

因此：

$$
\boxed{
\operatorname{softmax}(z)
=
\operatorname{softmax}(z-c)
}
$$

只要同一行的所有 logits 都减去同一个 `c`，softmax 结果完全不变。

最有用的选择就是：

$$
\boxed{c=\max_j z_j}.
$$

这样 shifted logits：

$$
z'_i=z_i-\max_j z_j
$$

满足：

$$
z'_i\le0.
$$

所以最大的 exponential 是：

$$
e^0=1,
$$

其他 exponential 都不超过 1。

这就避免了对巨大正数直接做 exponential 的 overflow 风险。


## 3. `[1000,999,998]`：减 Maximum 后，问题瞬间变小

原 logits：

$$
z=[1000,999,998].
$$

行内最大值：

$$
m=1000.
$$

减去最大值：

$$
z-m=[0,-1,-2].
$$

因为 softmax shift-invariant：

$$
\operatorname{softmax}([1000,999,998])
=
\operatorname{softmax}([0,-1,-2]).
$$

现在只需要计算：

$$
[e^0,e^{-1},e^{-2}]
\approx
[1,0.3679,0.1353].
$$

总和约：

$$
1.5032.
$$

所以：

$$
\operatorname{softmax}(z)
\approx
[0.6652,0.2447,0.0900].
$$

与 Step 11.4 的 `[2,1,0]` 例子得到同样结果并不是巧合，因为：

$$
[1000,999,998]
=
[2,1,0]+998.
$$

Softmax 只关心同一行内部的相对 differences，不关心整行共同平移了多少。


In [ ]:
import mlx.core as mx

logits = mx.array([1000.0, 999.0, 998.0])

# Naive formula: mathematically correct, numerically risky.
naive_exp = mx.exp(logits)
naive_softmax = naive_exp / mx.sum(naive_exp)

# Stable formula: subtract the row maximum first.
row_max = mx.max(logits)
shifted = logits - row_max
stable_exp = mx.exp(shifted)
stable_softmax = stable_exp / mx.sum(stable_exp)

# Framework operation: use this in real model code.
mlx_softmax = mx.softmax(logits, axis=-1)

print('naive exp:', naive_exp)
print('naive softmax:', naive_softmax)
print('shifted logits:', shifted)
print('manual stable softmax:', stable_softmax)
print('mx.softmax:', mlx_softmax)


## 4. Underflow 为什么通常比 Overflow 更容易接受？

减去最大值以后，所有 logits 都不大于 0。

如果某个位置远小于最大值，例如：

$$
z'_j=-1000,
$$

那么：

$$
e^{-1000}
$$

在有限精度中可能 underflow 到 0。

这与 overflow 的后果不同。

如果一个位置相对最大 logit 已经小了极多，它的理论 softmax probability 本来就极接近 0；数值上近似成 0 通常与其实际贡献一致。

而 overflow 可能产生 `inf/inf -> nan`，会直接污染整行输出。

因此 subtract-max 的核心目的，是把危险的巨大正 exponential 转换成最多为 1 的 exponential，同时允许非常不重要的位置自然趋近 0。

这里不是说 underflow 在所有数值计算里都“永远无害”，只是对这种 softmax probability 场景，它通常比 positive-exponential overflow 更可控。


## 5. `-∞` Causal Mask 和 Stable Softmax 能一起工作吗？

可以，而且逻辑非常自然。

例如一个 masked row：

$$
z=[5,-\infty,-\infty].
$$

行内最大值：

$$
m=5.
$$

subtract max：

$$
z-m
=
[0,-\infty,-\infty].
$$

然后：

$$
e^{z-m}
=
[1,0,0].
$$

所以：

$$
\operatorname{softmax}(z)
=
[1,0,0].
$$

mask 的语义完整保留。

### 为什么要保证每一行至少一个合法位置？

如果整行都是：

$$
[-\infty,-\infty,\ldots],
$$

那么 row max 也是 `-∞`，接下来会遇到：

$$
-\infty-(-\infty),
$$

这不是有效的有限差值。

Step 12.2 的 causal mask 保留 diagonal `j=i`，因此标准 causal self-attention 每个 Query row 至少有一个合法 finite score，不会形成全 mask 行。


## 6. Scaling 到底帮助什么？它和 Stable Softmax 不是同一件事

Step 11.4 已经解释：在独立、零均值、单位方差的教学假设下：

$$
\mathrm{Std}(q\cdot k)
\approx
\sqrt{d_k}.
$$

所以使用：

$$
\frac{q\cdot k}{\sqrt{d_k}}
$$

主要是让不同 `d_k` 下 raw logits 的典型尺度更可控。

这会带来两个相关但不同的好处。

### A. 避免 Softmax 过早变得极度 Peaked

如果 logits 间差距很大，softmax 很容易接近 one-hot。

例如：

```text
大尺度 logits
→ 最大位置接近 1
→ 其他位置接近 0
```

Scaling 可以让训练早期的 logits 不那么容易因为维度本身而被放大，使 attention distribution 的动态范围更可控。

### B. 改善 Optimization Conditions

当 softmax 极度饱和时，很多非最大位置的 probability 已非常接近 0，它们对应的局部敏感度也会很小；这会让参数更新更加困难。

所以 scaling 与 gradient behavior 有关系，但准确说法不是：

> “Scaling 只是为了防止 gradient vanishing。”

更准确的是：

> **Scaling 控制 dot-product logits 的典型尺度，减少 softmax 因维度增长而过度饱和，从而改善数值行为与优化条件。**

而 subtract-max stable softmax 解决的是另一个问题：

> **即使 logits 本身很大，也避免 exponential overflow。**

所以二者职责不要混在一起。


## 7. MLX 0.32.2：实际代码应该直接使用 `mx.softmax`

项目固定：

```text
mlx==0.32.2
```

该版本 `mx.softmax` 的 API：

```python
mx.softmax(a, axis=..., precise=False)
```

官方接口将它定义为 `exp(a)/sum(exp(a))` 的 **numerically stable version**。

因此实际 Attention 代码应写：

```python
attention_weights = mx.softmax(masked_scores, axis=-1)
```

而不是自己写朴素：

```python
mx.exp(masked_scores) / mx.sum(mx.exp(masked_scores), axis=-1)
```

### `precise=True` 是什么？

MLX 0.32.2 还提供：

```python
mx.softmax(a, axis=-1, precise=True)
```

对于低精度输入，`precise=True` 会用 float32 做 reduction accumulation；默认 `False` 时 accumulation type 跟随输入 dtype。

这属于 API 数值精度选项，不改变 softmax 的数学定义。本课程当前不把它扩展成 mixed-precision 专题。


In [ ]:
# A masked-score example; no full attention head is built here.
masked_scores = mx.array([
    [4.0, float('-inf'), float('-inf')],
    [3.0, 2.0, float('-inf')],
    [1000.0, 999.0, 998.0],
])

# Framework stable softmax along the Key axis.
weights = mx.softmax(masked_scores, axis=-1)

print('masked_scores:')
print(masked_scores)
print('weights:')
print(weights)
print('row sums:', mx.sum(weights, axis=-1))


## 8. Step 12.3 到这里的完整逻辑

Attention score pipeline 现在可以准确拆成三个数值职责：

```text
QK^T
↓
/ sqrt(d_k)
控制 dot-product logits 的典型尺度
↓
+ causal mask
禁止 future positions
↓
stable softmax over Key axis
把合法 logits 变成 normalized attention weights
```

数学上：

$$
S_{masked}
=
\frac{QK^T}{\sqrt{d_k}}+M,
$$

然后：

$$
\boxed{
A
=
\operatorname{softmax}(S_{masked},\ axis=-1)
}.
$$

Stable softmax 可以理解为利用：

$$
\operatorname{softmax}(z)
=
\operatorname{softmax}(z-\max(z))
$$

把最大 shifted logit 变成 0，从而让 exponentials 不超过 1。

### 本节边界

- naive exponential softmax 可能因巨大正 logits overflow；subtract-row-max 保持数学结果不变，却显著改善数值稳定性。
- 很负的 shifted logits 可能 underflow 到 0；对本来就极小的 softmax probability，这通常比 overflow/NaN 更可控。
- causal `-inf` mask 与 stable softmax兼容，只要每行至少有一个合法 finite score；标准 causal diagonal 保证这一点。
- scaling 与 stable-softmax 是不同机制：前者控制典型 logit scale / softmax saturation，后者避免 exponential overflow。
- 不应把 `1/sqrt(d_k)` 简化成“只是为了防止梯度消失”；它更准确地服务于 score scale、softmax regime 与 optimization conditions。
- MLX 0.32.2 的 `mx.softmax` 是 numerically stable operation；实际模型代码直接使用它。
- 本轮只准备 Step 12.3；不进入 Step 12.4，不实现完整 causal attention head。
